# 逐字 Verbatim — 雲端轉錄引擎
按 **Ctrl+F9**（Mac：⌘+F9）全部執行。跳出「這個筆記本並非由 Google 撰寫」時選 **仍要執行**。
完成後點橘色按鈕回到網站，這個分頁保持開著就好。

In [ ]:
#@title ▶ 啟動逐字 Verbatim　　（按 Ctrl+F9 或左邊的播放鍵，約 2 分鐘）
SITE = "https://bevis0414-crypto.github.io/verbatim/"
RAW = "https://raw.githubusercontent.com/bevis0414-crypto/verbatim/main/server.py"
import os, re, subprocess, sys, time, threading, urllib.request
from IPython.display import HTML, display, clear_output

def say(msg, color="#8b9098"):
    display(HTML(f'<div style="font:15px sans-serif;color:{color};margin:4px 0">{msg}</div>'))

def alive(url):
    try:
        urllib.request.urlopen(url + "/api/system", timeout=5); return True
    except Exception:
        return False

if subprocess.run("nvidia-smi", shell=True, capture_output=True).returncode != 0:
    say("⚠️ 目前沒有 GPU，轉錄會非常慢。請到上方選單「執行階段 → 變更執行階段類型」選 T4 GPU，再重新執行。", "#ff5a1f")

say("① 安裝套件中…")
subprocess.run(f"{sys.executable} -m pip install -q 'faster-whisper>=1.1.1' opencc-python-reimplemented python-multipart fastapi uvicorn speechbrain nvidia-cublas-cu12 'nvidia-cudnn-cu12>=9,<10'", shell=True)
# faster-whisper 需要 CUDA 12 版的 cuBLAS／cuDNN；Colab 內建的版本可能較新，所以另外指定
import glob, site
libdirs = []
for sp in site.getsitepackages():
    for pat in ("nvidia/cublas/lib", "nvidia/cudnn/lib", "nvidia/cuda_nvrtc/lib", "nvidia/cuda_runtime/lib"):
        libdirs += glob.glob(os.path.join(sp, pat))
os.makedirs("/content/verbatim", exist_ok=True)
urllib.request.urlretrieve(f"{RAW}?t={int(time.time())}", "/content/verbatim/server.py")  # 每次都抓最新版
if not os.path.exists("/content/cloudflared"):
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "/content/cloudflared")
    os.chmod("/content/cloudflared", 0o755)

say("② 啟動轉錄引擎…")
subprocess.run("pkill -f /content/verbatim/server.py", shell=True)  # 關掉舊版引擎
time.sleep(1)
env = dict(os.environ, PORT="8000", VERBATIM_PRELOAD="large-v2",
           LD_LIBRARY_PATH=":".join(libdirs + [os.environ.get("LD_LIBRARY_PATH", "")]))
subprocess.Popen([sys.executable, "/content/verbatim/server.py"], env=env, cwd="/content/verbatim",
                 stdout=open("/content/verbatim/log.txt", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    if alive("http://localhost:8000"): break
    time.sleep(1)
else:
    say("引擎啟動失敗，最後幾行紀錄：<pre>" + open("/content/verbatim/log.txt").read()[-1500:] + "</pre>", "#ff5a1f")
    raise SystemExit

say("③ 載入 Large v2 模型（第一次約 1–2 分鐘）…")
import json as _json
st = {}
for _ in range(240):
    try:
        st = _json.load(urllib.request.urlopen("http://localhost:8000/api/system", timeout=5))
        if st.get("loaded") or st.get("note"): break
    except Exception:
        pass
    time.sleep(1)
if st.get("note"):
    say("⚠️ " + st["note"], "#ff5a1f")

say("④ 建立連線網址…")
url = globals().get("_verbatim_url")
if not (url and alive(url)):
    subprocess.run("pkill -f cloudflared", shell=True)
    proc = subprocess.Popen(["/content/cloudflared", "tunnel", "--no-autoupdate", "--url", "http://localhost:8000"],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    url = None
    for line in proc.stdout:
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
        if m:
            url = m.group(0); break
    threading.Thread(target=lambda: [None for _ in proc.stdout], daemon=True).start()
    for _ in range(30):
        if alive(url): break
        time.sleep(2)
    _verbatim_url = url

clear_output()
link = SITE + "#server=" + url
display(HTML(f"""
<div style="font-family:sans-serif;background:#15171b;color:#eceae4;border-radius:16px;padding:28px;max-width:640px">
  <div style="font-size:22px;font-weight:700;margin-bottom:6px">✅ 轉錄引擎已啟動</div>
  <div style="color:#8b9098;margin-bottom:20px">點下面的按鈕回到逐字網站。<b style="color:#eceae4">這個 Colab 分頁請保持開著</b>，關掉後網站就會斷線。</div>
  <a href="{link}" target="_blank" style="display:inline-block;background:#ff5a1f;color:#fff;padding:14px 28px;border-radius:12px;font-size:17px;font-weight:600;text-decoration:none">回到逐字網站 ↗</a>
  {('<div style="color:#ff5a1f;font-size:14px;margin-top:16px">⚠️ ' + st["note"] + '</div>') if st.get("note") else ''}
  <div style="color:#8b9098;font-size:13px;margin-top:18px">連線網址：{url}</div>
</div>"""))
